# Lesson 26: Time Series Forecasting

*ML Academy — companion notebook*

Trend and seasonality, baselines that are hard to beat, lag features, backtesting, MASE, and forecasting several steps ahead.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-26.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Trend, seasonality and noise

In [1]:
import numpy as np
import pandas as pd

# Three years of simulated daily sales for a shop: trend + weekly pattern + yearly season + noise
rng = np.random.default_rng(0)
days = pd.date_range("2023-01-01", periods=3 * 365, freq="D")
t = np.arange(len(days))
weekly = np.array([0.9, 0.95, 1.0, 1.0, 1.1, 1.35, 1.25])[days.dayofweek]       # Mon … Sun
yearly = 1 + 0.2 * np.sin(2 * np.pi * (days.dayofyear - 80) / 365)
sales = (200 + 0.15 * t) * weekly * yearly * rng.lognormal(0, 0.08, len(t))
s = pd.Series(sales.round(1), index=days, name="sales")
print(s.head(7).to_string())
print("average sales by weekday (Mon … Sun):", s.groupby(s.index.dayofweek).mean().round(0).tolist())
print("average sales by year:", s.groupby(s.index.year).mean().round(0).tolist())

2023-01-01    203.1
2023-01-02    143.5
2023-01-03    161.4
2023-01-04    163.1
2023-01-05    155.2
2023-01-06    183.8
2023-01-07    243.8
Freq: D
average sales by weekday (Mon … Sun): [254.0, 267.0, 282.0, 281.0, 308.0, 383.0, 349.0]
average sales by year: [245.0, 304.0, 361.0]


## 3. Baselines first

In [2]:
# Forecast the last 8 weeks with simple baselines, one day ahead each day
test = s[-56:]
naive = s.shift(1)[-56:]                         # tomorrow = today
seasonal = s.shift(7)[-56:]                      # tomorrow = same weekday last week
moving = s.shift(1).rolling(28).mean()[-56:]     # tomorrow = average of the last 4 weeks
mae = lambda f: np.mean(np.abs(test - f))
for name, f in [("naive (yesterday)", naive), ("seasonal naive (last week)", seasonal), ("28-day average", moving)]:
    print(f"{name:27}: MAE {mae(f):6.1f}")

naive (yesterday)          : MAE   50.0
seasonal naive (last week) : MAE   32.6
28-day average             : MAE   45.3


## 5. Turning a series into features

In [3]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# Features that only use the past: lags, rolling means and calendar fields
df = pd.DataFrame({"y": s})
for lag in [1, 7, 14, 28]:
    df[f"lag_{lag}"] = s.shift(lag)
df["roll_7"] = s.shift(1).rolling(7).mean()
df["roll_28"] = s.shift(1).rolling(28).mean()
df["dow"] = df.index.dayofweek
df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * df.index.dayofyear / 365), np.cos(2 * np.pi * df.index.dayofyear / 365)
df = df.dropna()
Xf = pd.get_dummies(df.drop(columns="y"), columns=["dow"], dtype=float)
start = s.index[-56]                                                    # the last 56 days are the test period
train, test_idx = Xf.index < start, Xf.index >= start
yt = df.y[train].to_numpy()
scale = np.mean(np.abs(yt[7:] - yt[:-7]))                               # seasonal-naive MAE on training data, for MASE
for name, m in [("ridge on lag features", Ridge(1.0)), ("gradient boosting", HistGradientBoostingRegressor(random_state=0))]:
    pred = m.fit(Xf[train], df.y[train]).predict(Xf[test_idx])
    err = np.mean(np.abs(df.y[test_idx] - pred))
    print(f"{name:22}: MAE {err:6.1f}   MASE {err / scale:.2f}")

ridge on lag features : MAE   26.2   MASE 0.98


gradient boosting     : MAE   28.9   MASE 1.08


## 6. Validating forecasts: backtesting

In [4]:
from sklearn.model_selection import TimeSeriesSplit

# Backtesting: repeat "train on the past, test on the next 8 weeks" several times
maes = []
for tr, te in TimeSeriesSplit(n_splits=5, test_size=56).split(Xf):
    pred = Ridge(1.0).fit(Xf.iloc[tr], df.y.iloc[tr]).predict(Xf.iloc[te])
    maes.append(np.mean(np.abs(df.y.iloc[te] - pred)))
print("MAE in each backtest window:", np.round(maes, 1))
print("average:", round(np.mean(maes), 1))

MAE in each backtest window: [21.9 28.3 28.6 19.8 26.2]
average: 25.0


## 7. Forecasting several steps ahead

In [5]:
# Forecasting 14 days ahead recursively: each prediction becomes the next day's lag
history = s[s.index < start].copy()
model = Ridge(1.0).fit(Xf[train], df.y[train])
cols = Xf.columns
for step in range(14):
    day = history.index[-1] + pd.Timedelta(days=1)
    row = {f"lag_{l}": history.iloc[-l] for l in [1, 7, 14, 28]}
    row.update(roll_7=history.iloc[-7:].mean(), roll_28=history.iloc[-28:].mean(),
               doy_sin=np.sin(2 * np.pi * day.dayofyear / 365), doy_cos=np.cos(2 * np.pi * day.dayofyear / 365))
    row.update({f"dow_{d}": float(day.dayofweek == d) for d in range(7)})
    history.loc[day] = model.predict(pd.DataFrame([row])[cols])[0]
err = np.abs(history[-14:] - s[history.index[-14:]])
print("absolute error, days 1–7 ahead: ", round(err[:7].mean(), 1))
print("absolute error, days 8–14 ahead:", round(err[7:].mean(), 1))

absolute error, days 1–7 ahead:  28.1
absolute error, days 8–14 ahead: 24.8


## 9. In Python: a compact forecasting pipeline

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

rng = np.random.default_rng(1)
idx = pd.date_range("2024-01-01", periods=730, freq="D")
y = pd.Series((100 + 0.05 * np.arange(730)) * np.array([0.9, 1, 1, 1, 1.1, 1.3, 1.2])[idx.dayofweek]
              * rng.lognormal(0, 0.1, 730), index=idx)

def make_features(y):
    X = pd.DataFrame({f"lag_{l}": y.shift(l) for l in [1, 7, 14]}, index=y.index)
    X["roll_7"] = y.shift(1).rolling(7).mean()
    return X.join(pd.get_dummies(pd.Series(y.index.dayofweek, index=y.index), prefix="dow", dtype=float))

X = make_features(y).dropna(); target = y[X.index]
cut = X.index[-60]                                        # the last 60 days are the test period
model = Ridge(1.0).fit(X[X.index < cut], target[target.index < cut])
pred = model.predict(X[X.index >= cut])
actual, base = target[target.index >= cut], y.shift(7)[target.index[target.index >= cut]]
print("model MAE:", round(np.mean(np.abs(actual - pred)), 2), "  seasonal naive MAE:", round(np.mean(np.abs(actual - base)), 2))

model MAE: 14.52   seasonal naive MAE: 20.01


## 12. Going deeper: choosing lags and honest prediction intervals

In [7]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(3)
t = np.arange(730)
y = pd.Series((200 + 0.15 * t) * np.array([0.85, 0.9, 0.95, 1.0, 1.05, 1.25, 1.1])[t % 7] * rng.lognormal(0, 0.08, 730))
change = y.diff(1)                                   # remove the trend first
acf = {lag: round(float(change.autocorr(lag)), 2) for lag in [1, 2, 3, 6, 7, 14, 21]}
print("autocorrelation of daily changes by lag:", acf)

autocorrelation of daily changes by lag: {1: -0.12, 2: -0.32, 3: -0.07, 6: 0.07, 7: 0.61, 14: 0.59, 21: 0.6}


In [8]:
from sklearn.linear_model import Ridge
X = pd.DataFrame({f"lag_{l}": y.shift(l) for l in [7, 14, 21]}).dropna(); target = y[X.index]
cal, test = X.index[(X.index >= 550) & (X.index < 640)], X.index[X.index >= 640]
model = Ridge(1.0).fit(X[X.index < 550], target[target.index < 550])
resid = target[cal] - model.predict(X.loc[cal])
lo, hi = np.quantile(resid, [0.05, 0.95])            # 90% interval from held-out errors
pred = model.predict(X.loc[test])
inside = ((target[test] >= pred + lo) & (target[test] <= pred + hi)).mean()
print("interval offsets:", round(lo, 1), "to", round(hi, 1), "  coverage on the test period:", round(inside, 3))

interval offsets: -40.3 to 50.2   coverage on the test period: 0.856


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
rng = np.random.default_rng(0)
months = pd.date_range("2023-01-01", periods=36, freq="MS")
level = df.price_lakh.mean() * (1 + 0.006 * np.arange(36)) * (1 + 0.03 * np.sin(2 * np.pi * months.month / 12))
index = pd.Series(level * rng.normal(1, 0.01, 36), index=months).round(1)   # a simulated monthly price index
test = index.index >= "2025-07-01"
naive, seasonal = index.shift(1)[test], index.shift(12)[test]
print("last 6 months:", index[test].tolist())
print("MAE, last month:", round((index[test] - naive).abs().mean(), 2), "  MAE, same month last year:", round((index[test] - seasonal).abs().mean(), 2))

last 6 months: [82.9, 83.1, 83.2, 84.6, 85.7, 87.5]
MAE, last month: 1.08   MAE, same month last year: 4.77


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: The seasonal naive forecast

Forecast each of the last 7 days with the value from 7 days earlier, and compute the MAE.

In [11]:
import numpy as np
y = np.array([50, 52, 49, 70, 75, 51, 53, 54, 51, 50, 72, 78, 50, 55])
actual = y[7:]
forecast = actual  # ← replace with the values from 7 days earlier
result = round(float(np.mean(np.abs(actual - forecast))), 2)
check(result, 2.0, hint='The forecast for day i is y[i - 7].')

❌ Not yet. Hint: The forecast for day i is y[i - 7].


### Challenge 2: A past-only rolling mean

Make a 3-day rolling mean that only uses days before each day (shift first).

In [12]:
import pandas as pd
s = pd.Series([10, 12, 14, 16, 18])
roll = s.rolling(3).mean()  # ← this includes the current day; fix it
result = roll.tolist()[3:]
check(result, [12.0, 14.0], hint='Shift by one, then roll.')

❌ Not yet. Hint: Shift by one, then roll.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. The seasonal naive forecast** — the missing line:

```python
forecast = y[:7]
```

**2. A past-only rolling mean** — the missing line:

```python
roll = s.shift(1).rolling(3).mean()
```